# Digital Me: AI Persona & RAG Digital Twin for Pranith Kumar Dupuguntla
### Autonomous Knowledge Retrieval, Vector Store Embedding & Persona Generation

This notebook implements an end-to-end **Retrieval-Augmented Generation (RAG)** pipeline representing **Pranith Kumar Dupuguntla**.
It ingests personal records, resume summaries, the 41-page **ByteLife** mini-project report, and the published **IJIRSET** research paper, indexes them in a **ChromaDB** vector store using **FastEmbed** (`BAAI/bge-small-en-v1.5`), and responds to questions in the first person.

In [1]:
# Step 1: Import Dependencies
import os
import sys
import glob
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.embeddings import FastEmbedEmbeddings
from langchain_chroma import Chroma

print("All core LangChain, FastEmbed, and Chroma modules imported successfully!")

C:\Users\prane\AppData\Local\Temp\ipykernel_3608\641405860.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import FastEmbedEmbeddings


All core LangChain, FastEmbed, and Chroma modules imported successfully!


## Step 2: Load Knowledge Base Documents
We load all 5 foundational documents:
1. `personal_qa.txt`: Personal Q&A, education, and career aspirations.
2. `resume_summary.txt`: Power BI, Excel, and analytics competencies.
3. `projects_research.txt`: High-level summary of academic accomplishments.
4. `bytelife_report.txt`: Full 41-page mini project report (Random Forest, clinical safety rules, telemetry).
5. `microskills_paper.txt`: Full 8-page IJIRSET published research paper on AI resume & micro-skill extraction.

In [2]:
# Step 2: Load and Inspect Documents
KB_DIR = os.path.join(os.getcwd(), "knowledge_base")
if not os.path.exists(KB_DIR):
    KB_DIR = os.getcwd()

documents = []
txt_files = glob.glob(os.path.join(KB_DIR, "*.txt"))

for fpath in txt_files:
    fname = os.path.basename(fpath)
    with open(fpath, "r", encoding="utf-8", errors="replace") as f:
        content = f.read()
        documents.append({"source": fname, "content": content})
    print(f"Loaded: {fname} ({len(content)} characters)")

print(f"\nSuccessfully loaded {len(documents)} knowledge base documents.")

Loaded: bytelife_report.txt (46657 characters)
Loaded: microskills_paper.txt (12261 characters)
Loaded: personal_qa.txt (1507 characters)
Loaded: projects_research.txt (1261 characters)
Loaded: resume_summary.txt (1589 characters)

Successfully loaded 5 knowledge base documents.


## Step 3: Text Chunking with RecursiveCharacterTextSplitter
We split the source documents into semantic chunks with 550 characters and 70 overlap to preserve contextual continuity.

In [3]:
# Step 3: Chunk Documents
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=550,
    chunk_overlap=70,
    separators=["\n\n--- [Page", "\n\n", "\n", " ", ""]
)

all_chunks = []
for doc in documents:
    chunks = text_splitter.split_text(doc["content"])
    for i, chunk in enumerate(chunks):
        all_chunks.append(Document(
            page_content=chunk.strip(),
            metadata={
                "source": doc["source"],
                "chunk_id": f"{doc['source']}_chunk_{i}"
            }
        ))
    print(f"- {doc['source']}: {len(chunks)} chunks")

print(f"\nTotal chunks ready for vector embeddings: {len(all_chunks)}")

- bytelife_report.txt: 110 chunks
- microskills_paper.txt: 29 chunks
- personal_qa.txt: 4 chunks
- projects_research.txt: 5 chunks
- resume_summary.txt: 4 chunks

Total chunks ready for vector embeddings: 152


## Step 4: FastEmbed Embedding & Persistent ChromaDB Index
We initialize FastEmbed (`BAAI/bge-small-en-v1.5`), which generates dense vector representations without PyTorch overhead or DLL issues on Windows.

In [4]:
# Step 4: Initialize FastEmbed and ChromaDB
embeddings = FastEmbedEmbeddings(model_name="BAAI/bge-small-en-v1.5")
PERSIST_DIR = os.path.join(os.getcwd(), "chroma_digital_me")

vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    persist_directory=PERSIST_DIR
)

print(f"Vector database successfully indexed and saved at: {PERSIST_DIR}")

C:\Users\prane\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Vector database successfully indexed and saved at: C:\Users\prane\OneDrive\Desktop\digit_me doc\chroma_digital_me


## Step 5: Test Semantic Similarity Search

In [5]:
# Step 5: Semantic Retrieval Test
test_queries = [
    "What projects has Pranith built?",
    "Explain the Random Forest classifier and clinical overrides in ByteLife",
    "What is the title and journal of Pranith's published research paper?",
    "What Power BI and DAX skills does Pranith possess?"
]

for q in test_queries:
    print(f"\n=== Query: '{q}' ===")
    hits = vectorstore.similarity_search(q, k=2)
    for idx, hit in enumerate(hits, 1):
        snippet = hit.page_content.replace("\n", " ")[:160]
        print(f"  [{idx}] ({hit.metadata.get('source')}): {snippet}...")


=== Query: 'What projects has Pranith built?' ===
  [1] (personal_qa.txt): Q: What are some key projects you have built? A: I have built several projects, including: - An interactive Hospital Management Dashboard and Employee Performan...
  [2] (personal_qa.txt): Q: What are some key projects you have built? A: I have built several projects, including: - An interactive Hospital Management Dashboard and Employee Performan...

=== Query: 'Explain the Random Forest classifier and clinical overrides in ByteLife' ===
  [1] (projects_research.txt): - Functionality: Uses a Machine Learning model (Random Forest / Hybrid classifier) to predict normal, abnormal, or critical health conditions and automatically ...
  [2] (projects_research.txt): - Functionality: Uses a Machine Learning model (Random Forest / Hybrid classifier) to predict normal, abnormal, or critical health conditions and automatically ...

=== Query: 'What is the title and journal of Pranith's published research paper?' ===
  [1

## Step 6: Persona System Prompt & Generation Engine
We condition the model with a first-person persona system prompt so it speaks as Pranith Kumar Dupuguntla.

In [6]:
# Step 6: Digital Me Persona Prompt and Synthesis Pipeline
from digital_me_engine import ask_digital_me

def chat_with_pranith(user_question: str):
    """Ask Digital Pranith a question and print the formatted first-person response."""
    result = ask_digital_me(user_question)
    print(f"User: {user_question}\n")
    print(f"Digital Pranith:\n{result['response']}\n")
    print("Retrieved Sources:")
    for s in result["sources"]:
        print(f" - {s['title']} ({s['source']})")
    print("-" * 60)
    return result

# Live Test 1: Project overview
chat_with_pranith("Tell me about your ByteLife healthcare project and its machine learning model.")

User: Tell me about your ByteLife healthcare project and its machine learning model.

Digital Pranith:
### 💓 Early Detection & Intervention for the Aging Population (ByteLife)

I co-authored and engineered **ByteLife**, an end-to-end telemetry and emergency dispatch platform specifically designed for elderly individuals living independently.

**Key Architectural Highlights**:
- **Physiological Telemetry**: Captures 5 vital parameters in real time: Heart Rate (bpm), Systolic & Diastolic Blood Pressure (mmHg), Oxygen Saturation (SpO₂ in %), and Body Temperature (°F).
- **Dual-Layer Intelligence**: Combines a **Random Forest Classifier (99.78% accuracy)** with a deterministic **Clinical Rule-Based Failsafe Override** (e.g., promoting patients immediately to *Critical* if SpO₂ drops below 90% or Systolic BP exceeds 180 mmHg).
- **Automated Alerting**: Instantly dispatches rich HTML emails to family caretakers with patient vitals, deviation flags, and actionable first-aid / OTC medicine gui

{'query': 'Tell me about your ByteLife healthcare project and its machine learning model.',
 'response': '### 💓 Early Detection & Intervention for the Aging Population (ByteLife)\n\nI co-authored and engineered **ByteLife**, an end-to-end telemetry and emergency dispatch platform specifically designed for elderly individuals living independently.\n\n**Key Architectural Highlights**:\n- **Physiological Telemetry**: Captures 5 vital parameters in real time: Heart Rate (bpm), Systolic & Diastolic Blood Pressure (mmHg), Oxygen Saturation (SpO₂ in %), and Body Temperature (°F).\n- **Dual-Layer Intelligence**: Combines a **Random Forest Classifier (99.78% accuracy)** with a deterministic **Clinical Rule-Based Failsafe Override** (e.g., promoting patients immediately to *Critical* if SpO₂ drops below 90% or Systolic BP exceeds 180 mmHg).\n- **Automated Alerting**: Instantly dispatches rich HTML emails to family caretakers with patient vitals, deviation flags, and actionable first-aid / OTC me

In [7]:
# Live Test 2: Research Publication
chat_with_pranith("Where did you publish your research paper and what problem does it solve?")

User: Where did you publish your research paper and what problem does it solve?

Digital Pranith:
### 📄 Published Research in IJIRSET (March 2026)

I co-authored a peer-reviewed research paper titled **"Detecting Micro Skills and Unlocking Macro Careers"**, published in the *International Journal of Innovative Research in Science, Engineering and Technology* (IJIRSET, Vol. 15, Issue 3, Impact Factor: 8.699, DOI: 10.15680/IJIRSET.2026.1503130).

**Core Contribution & Methodology**:
- Solved the fundamental limitations of traditional keyword-based ATS resume filters.
- Built an AI-driven platform using **Natural Language Processing (NLP)** and Machine Learning to parse resumes and extract verified micro-skills from candidate coding repositories (GitHub, LeetCode).
- Performs automated skill-gap analysis, generates ATS-optimized resume recommendations, and charts personalized upskilling roadmaps mapped to industry roles.

Retrieved Sources:
 - Projects & Research Summary (projects_researc

{'query': 'Where did you publish your research paper and what problem does it solve?',
 'response': '### 📄 Published Research in IJIRSET (March 2026)\n\nI co-authored a peer-reviewed research paper titled **"Detecting Micro Skills and Unlocking Macro Careers"**, published in the *International Journal of Innovative Research in Science, Engineering and Technology* (IJIRSET, Vol. 15, Issue 3, Impact Factor: 8.699, DOI: 10.15680/IJIRSET.2026.1503130).\n\n**Core Contribution & Methodology**:\n- Solved the fundamental limitations of traditional keyword-based ATS resume filters.\n- Built an AI-driven platform using **Natural Language Processing (NLP)** and Machine Learning to parse resumes and extract verified micro-skills from candidate coding repositories (GitHub, LeetCode).\n- Performs automated skill-gap analysis, generates ATS-optimized resume recommendations, and charts personalized upskilling roadmaps mapped to industry roles.',
 'sources': [{'title': 'Projects & Research Summary',
  

In [8]:
# Live Test 3: Technical Skills & Contact
chat_with_pranith("What are your core technical skills in Power BI and how can I contact you?")

User: What are your core technical skills in Power BI and how can I contact you?

Digital Pranith:
### 🛠️ Technical Competencies & Tooling

My primary data analytics toolkit includes:

- **Business Intelligence**: **Power BI** (Advanced DAX formulas, Power Query M-code, star-schema data modeling, interactive KPI dashboards) and **Tableau**.
- **Spreadsheet Engineering**: **Microsoft Excel** (XLOOKUP, INDEX-MATCH, Pivot Tables, Pivot Charts, Conditional Formatting, automated reporting models).
- **Programming & Databases**: **Python** (Pandas, NumPy, Scikit-Learn, Matplotlib, Seaborn, OpenCV, MediaPipe), **SQL** (PostgreSQL, MySQL queries, joins, window functions), C++, and Java.
- **Core Domains**: ETL pipeline design, exploratory data analysis (EDA), predictive modeling, trend forecasting, and data-driven storytelling.

Retrieved Sources:
 - Professional Resume Summary (resume_summary.txt)
 - Professional Resume Summary (resume_summary.txt)
 - resume_summary.txt (resume_summary.txt)
-

{'query': 'What are your core technical skills in Power BI and how can I contact you?',
 'response': '### 🛠️ Technical Competencies & Tooling\n\nMy primary data analytics toolkit includes:\n\n- **Business Intelligence**: **Power BI** (Advanced DAX formulas, Power Query M-code, star-schema data modeling, interactive KPI dashboards) and **Tableau**.\n- **Spreadsheet Engineering**: **Microsoft Excel** (XLOOKUP, INDEX-MATCH, Pivot Tables, Pivot Charts, Conditional Formatting, automated reporting models).\n- **Programming & Databases**: **Python** (Pandas, NumPy, Scikit-Learn, Matplotlib, Seaborn, OpenCV, MediaPipe), **SQL** (PostgreSQL, MySQL queries, joins, window functions), C++, and Java.\n- **Core Domains**: ETL pipeline design, exploratory data analysis (EDA), predictive modeling, trend forecasting, and data-driven storytelling.',
 'sources': [{'title': 'Professional Resume Summary',
   'source': 'resume_summary.txt',
   'snippet': 'Technical Skills:\n- Power BI: DAX, Power Query, Dat